## Information Retrieval 24/25, University of Pisa
### Franco Maria Nardini, Rossano Venturini (francomaria.nardini@isti.cnr.it, rossano.venturini@unipi.it)

----

# Inverted Index

---
![image](imgs/common_crawl.png)

[Common Crawl](https://commoncrawl.org/) maintains a free, open repository of web crawl data that can be used by anyone.

- Over 250 billion pages spanning 17 years
- Free and open corpus since 2007
- A snapshot of 3–5 billion new pages are added each month


---

## C4 Dataset

[C4 (Colossal Clean Crawled Corpus) Dataset](https://huggingface.co/datasets/allenai/c4)

A colossal, cleaned version of Common Crawl's web crawl corpus (from Google). Based on Common Crawl dataset: "https://commoncrawl.org".

We use the processed version of Google's C4 dataset by Allen Institute for AI.
They prepared five variants of the data: `en`, `en.noclean`, `en.noblocklist`, `realnewslike`, and `multilingual (mC4)`.

For reference, these are the sizes of the variants:

- `en`: 305GB
- `en.noclean`: 2.3TB
- `en.noblocklist`: 380GB
- `realnewslike`: 15GB
- `multilingual (mC4)`: 9.7TB (108 subsets, one per language)

The `en.noblocklist` variant is exactly the same as the en variant, except we turned off the so-called "badwords filter", which removes all documents that contain words from the lists at https://github.com/LDNOOBW/List-of-Dirty-Naughty-Obscene-and-Otherwise-Bad-Words.

In [ ]:
!pip install datasets nltk unidecode tqdm

<br><br>

#### We download 4 out of 1024 files of size ~318Mb compressed each

In [49]:
from datasets import load_dataset

c4_subset = load_dataset("allenai/c4", data_files="en/c4-train.0102*-of-01024.json.gz")

<br><br>
#### Get a list of URLs and a list of corresponding documents

In [50]:
urls = [x['url'] for x in c4_subset["train"]]
documents = [x['text'] for doc_id, x in enumerate(c4_subset["train"])]

print(f"Number of documents: {len(urls)}")
print(f"Number of characters: {sum(len(x) for x in documents)} ({sum(len(x) for x in documents)/1024**2})") 

KeyboardInterrupt: 

In [51]:
c4_subset = None

In [5]:
print(urls[0])
print(documents[0][:500], "[...]")

https://americanhealthandbeauty.com/articles/2704/non-surgical-fat-reduction--zerona-vs-zeltiq
Liposuction has remained one of the most popular cosmetic surgeries for years as people turn to their doctors to remove the fat that diet and exercise can't seem to touch. Recently, there has been a trend towards less invasive aesthetic options as lasers and fillers replace facelifts and laser lipo takes center stage with traditional liposuction. There are two devices, both currently undergoing FDA testing, which could replace fat reduction surgery altogether. They are Zerona and Zeltiq, and the [...]


----

## Inverted Index

<br><br>

#### Let's build a simple inverted index



In [6]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
from unidecode import unidecode
from tqdm import tqdm
import math

# Download necessary data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\carmi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\carmi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\carmi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [7]:
def get_tokens(doc):   
    # Step 1: Convert document to lowercase
    doc = doc.lower()

    # Step 2: # Step 1: Normalize accents e.g., café vs cafe
    doc = unidecode(doc)
    
    # Step 3: Normalize multiple spaces to a single space
    doc = " ".join(doc.split())
    
    # Step 4: Tokenize the document
    tokens = word_tokenize(doc)
    
    # Step 5: Remove punctuation
    tokens_no_punct = [word for word in tokens if word not in string.punctuation]
    
    # Step 6: Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens_no_stopwords = [word for word in tokens_no_punct if word not in stop_words]
    
    # Step 7: Lemmatization (or stemming)
    #lemmatizer = WordNetLemmatizer()
    #tokens_lemmatized = [lemmatizer.lemmatize(word) for word in tokens_no_stopwords]
    
    # Step 8: Remove numbers
    tokens_no_numbers = [word for word in tokens_no_stopwords if not word.isdigit()]

    return tokens_no_numbers
    

In [8]:
get_tokens(documents[2])[:10]

['brighter',
 'communities',
 'worldwide',
 'undertakes',
 'many',
 'forms',
 'fundraising',
 'raise',
 'funds',
 'programmes']

<br><br>

<img src="imgs/inverted_indexes.png" alt="alt text" width="700" />

In [119]:
vocabulary_map = {} # Map from token to its id
df = []             # for ith term, tf[i] stores the number of documents containing this term (red in the image above)
frequencies = []    # for ith term, frequencies[i] stores its frequency in the whole collection (green in the image above)

posting_lists = []  # list of lists. In real life, it is a big list with offsets. (Lists in Python are vectors!)
tf = []             # list of lists. For each document, the frequency of the term in the document

doc_lengths = []    # the length of each document

for doc_id, document in tqdm(enumerate(documents[:10000])):
    terms = get_tokens(document)

    doc_lengths.append(len(terms))
    
    for term in terms: 
        if term not in vocabulary_map: # new term
            vocabulary_map[term] = len(vocabulary_map)
            posting_lists.append([])
            tf.append([])
            frequencies.append(0)
            df.append(0)
            
        term_id = vocabulary_map[term]
        frequencies[term_id] += 1
        
        if len(posting_lists[term_id]) == 0 or posting_lists[term_id][-1] != doc_id: # avoid duplicated doc_ids within the same list
            posting_lists[term_id].append( doc_id )
            tf[term_id].append( 0 )
            df[term_id] += 1
            
        tf[term_id][-1] += 1

vocabulary_map

10000it [00:21, 471.70it/s]


{'liposuction': 0,
 'remained': 1,
 'one': 2,
 'popular': 3,
 'cosmetic': 4,
 'surgeries': 5,
 'years': 6,
 'people': 7,
 'turn': 8,
 'doctors': 9,
 'remove': 10,
 'fat': 11,
 'diet': 12,
 'exercise': 13,
 'ca': 14,
 "n't": 15,
 'seem': 16,
 'touch': 17,
 'recently': 18,
 'trend': 19,
 'towards': 20,
 'less': 21,
 'invasive': 22,
 'aesthetic': 23,
 'options': 24,
 'lasers': 25,
 'fillers': 26,
 'replace': 27,
 'facelifts': 28,
 'laser': 29,
 'lipo': 30,
 'takes': 31,
 'center': 32,
 'stage': 33,
 'traditional': 34,
 'two': 35,
 'devices': 36,
 'currently': 37,
 'undergoing': 38,
 'fda': 39,
 'testing': 40,
 'could': 41,
 'reduction': 42,
 'surgery': 43,
 'altogether': 44,
 'zerona': 45,
 'zeltiq': 46,
 'pain': 47,
 'free': 48,
 'treatments': 49,
 'cut': 50,
 'made': 51,
 'erchonia': 52,
 'exactly': 53,
 'new': 54,
 'concept': 55,
 'newport': 56,
 'beach': 57,
 'physician': 58,
 'dr.': 59,
 'thomas': 60,
 'barnes': 61,
 'using': 62,
 'low': 63,
 'level': 64,
 'therapy': 65,
 'device': 6

<br><br>
#### BM25

To create an inverted index with BM25 scores in Python, you can follow these steps:

1. **Understand BM25 Formula**: The BM25 score for a term $ t $ in a document $ d $ is calculated as:

$$
BM25(t, d) = IDF(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot (1 - b + b \cdot \frac{|d|}{\text{avgdl}})}
$$

Where:
- $ f(t, d) $ is the frequency of the term $ t $ in the document $ d $.
- $ |d| $ is the length of the document.
- $ \text{avgdl} $ is the average document length in the collection.
- $ k_1 $ and $ b $ are hyperparameters (commonly set as $ k_1 = 1.5 $ and $ b = 0.75 $).
- $ IDF(t) $ is the inverse document frequency of the term $ t $:

$$
IDF(t) = \log\left(\frac{N - df(t) + 0.5}{df(t) + 0.5} + 1\right)
$$

Where $ N $ is the total number of documents and $ df(t) $ is the document frequency of term $ t $ (number of documents containing $ t $).

In [135]:
# Function to compute IDF for a term
def compute_idf(df_t, N):
    return math.log((N - df_t + 0.5) / (df_t + 0.5) + 1)
    
N = len(df)

# We precompute the idf. Online it could be expensive for the computation of ln 
idf = [ compute_idf(df_t, N) for df_t in df ] 

N
idf


[10.805809169049445,
 7.466487191105378,
 3.5018313633260765,
 5.634189455547366,
 8.272112355092013,
 9.370724643760123,
 4.200511248101243,
 4.197808543753358,
 5.566180798850088,
 7.333842716499083,
 6.182799064933023,
 7.21675005021772,
 7.035349727943086,
 6.548779024550249,
 5.147721653655374,
 3.801108684010829,
 5.958477425911382,
 5.933669952207115,
 5.609524528066561,
 7.017084379965793,
 5.918472091297683,
 5.112077030246746,
 8.48342144875922,
 7.996406473686948,
 5.553535741002816,
 9.707196880381336,
 10.017351808685175,
 6.70813681673467,
 10.805809169049445,
 7.904387574966695,
 10.805809169049445,
 5.609524528066561,
 5.435171140921782,
 6.167204206975117,
 5.924523546981038,
 4.152946139696098,
 6.436361316582424,
 5.440768170458501,
 8.444955167931424,
 8.608584591713226,
 6.607104591503102,
 4.318125150564835,
 7.194891256405221,
 7.297253269066791,
 7.972595824993229,
 11.316634792815435,
 11.316634792815435,
 6.520844247218695,
 4.601251406480755,
 7.2167500502177

In [120]:
# Average document length
avgdl = sum(doc_lengths) / N

# Function to compute BM25 score for a term in a document
# - b is in [0,1]. Larger b favors shorter documents
def compute_bm25(term_id, doc_id, term_freq, idf, avgdl, k1 = 1.5, b = 0.75):
    idf_t = idf[term_id]
    doc_len = doc_lengths[doc_id]
    
    # BM25 formula
    numerator = term_freq * (k1 + 1)
    denominator = term_freq + k1 * (1 - b + b * (doc_len / avgdl))
    
    return idf_t * (numerator/denominator)

<br>

----

#### Save on Disk

In [121]:
import pickle

In [122]:
filename = 'inverted_index.pkl' 

In [123]:
with open(filename, 'wb') as file:
    pickle.dump((N, avgdl, vocabulary_map, posting_lists, df, idf,  frequencies, tf, doc_lengths), file)

#### Read from Disk

In [124]:
with open(filename, 'rb') as file:
    N, avgdl, vocabulary_map, posting_lists, df, idf, frequencies, tf, doc_lengths = pickle.load(file)

----

#### Checks

In [125]:
frequencies[:5]

[3, 75, 9006, 548, 50]

In [126]:
len(frequencies)

assert len(frequencies) == len(df),  "both must have one element per term"
assert len(frequencies) == len(tf),  "both must have one element per term"

In [127]:
sorted(vocabulary_map.items(), key = lambda x: x[1])[:5]

[('liposuction', 0),
 ('remained', 1),
 ('one', 2),
 ('popular', 3),
 ('cosmetic', 4)]

In [128]:
term = "one"

term_id = vocabulary_map[term]
#posting_lists[term_id]
#list( zip(posting_lists[term_id], tf[term_id]) )[:5]
posting_lists[term_id]
tf[term_id]

[3,
 2,
 1,
 1,
 2,
 2,
 1,
 3,
 1,
 1,
 3,
 1,
 4,
 1,
 3,
 7,
 2,
 1,
 5,
 2,
 3,
 3,
 1,
 7,
 4,
 3,
 4,
 1,
 1,
 1,
 2,
 1,
 2,
 20,
 1,
 1,
 2,
 1,
 7,
 3,
 1,
 1,
 1,
 1,
 4,
 1,
 1,
 1,
 1,
 4,
 1,
 1,
 2,
 4,
 1,
 1,
 2,
 1,
 4,
 1,
 3,
 2,
 1,
 1,
 1,
 3,
 2,
 3,
 2,
 2,
 4,
 2,
 2,
 2,
 1,
 1,
 1,
 4,
 3,
 2,
 2,
 3,
 4,
 3,
 1,
 2,
 3,
 2,
 8,
 2,
 1,
 1,
 1,
 1,
 1,
 2,
 13,
 1,
 7,
 6,
 1,
 1,
 1,
 8,
 1,
 2,
 5,
 2,
 2,
 1,
 3,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 2,
 6,
 3,
 1,
 3,
 1,
 2,
 2,
 1,
 3,
 2,
 1,
 2,
 1,
 1,
 1,
 1,
 2,
 1,
 1,
 1,
 2,
 6,
 1,
 2,
 2,
 2,
 1,
 1,
 1,
 14,
 1,
 1,
 3,
 1,
 1,
 1,
 3,
 3,
 1,
 1,
 11,
 1,
 1,
 6,
 2,
 1,
 9,
 1,
 1,
 1,
 1,
 1,
 3,
 1,
 2,
 11,
 3,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 2,
 1,
 3,
 1,
 6,
 1,
 3,
 19,
 1,
 2,
 2,
 2,
 1,
 4,
 1,
 1,
 1,
 2,
 1,
 2,
 2,
 1,
 1,
 2,
 2,
 1,
 3,
 1,
 2,
 3,
 3,
 2,
 2,
 1,
 2,
 1,
 7,
 2,
 6,
 1,
 1,
 2,
 2,
 1,
 1,
 1,
 6,
 2,
 1,
 7,
 2,
 1,
 1,
 1,
 1,
 1,
 2,
 1,
 1,
 4,
 1,
 1,
 1,
 1,
 1

In [129]:
doc_id = posting_lists[term_id][0]
freq = tf[term_id][0]

print( freq, documents[doc_id].count(term) )

3 3


----

## Query Processing


<br><br>

### Term-at-a-Time (TAAT)


<img src="imgs/taat.png" alt="alt text" width="700" />


In [145]:
def taat_or(query):
    accumulators = {}
    
    for term in get_tokens(query):
        print(f"Processing term: {term}")
        if term not in vocabulary_map:
            print("Term not present")
            continue
        term_id = vocabulary_map[term]
        
        for doc_id, term_freq in zip(posting_lists[term_id], tf[term_id]):
            score = compute_bm25(term_id, doc_id, term_freq, idf, avgdl)
            if doc_id not in accumulators:
                accumulators[doc_id] = 0.0
            print(f"Term: {term}, add score {score} to doc_id {doc_id}")
            accumulators[doc_id] += score
            
    return sorted(accumulators.items(), key = lambda x: x[1], reverse = True)

In [137]:
# Print top documents with their scores
def print_top(top):
    print("\n\n")
    for doc_id, score in top:
        print(f"Score: {score}")
        print(documents[doc_id])
        print("\n\n" + "-"*50 + "\n\n")
        

In [146]:
top_5 = taat_or("Rust programming language")
print(len(top_5))
print_top(top_5)

Processing term: rust
Term: rust, add score 15.231620825546035 to doc_id 3594
Term: rust, add score 5.79089169070445 to doc_id 3920
Term: rust, add score 11.446970642082885 to doc_id 4161
Term: rust, add score 15.209898291363997 to doc_id 4506
Term: rust, add score 15.632026666446981 to doc_id 4922
Term: rust, add score 14.9854972709778 to doc_id 5143
Term: rust, add score 4.320115980667235 to doc_id 5376
Term: rust, add score 13.678549650817478 to doc_id 5986
Term: rust, add score 14.585851145537639 to doc_id 6325
Term: rust, add score 10.317433588226576 to doc_id 6867
Term: rust, add score 19.742960146187087 to doc_id 7090
Term: rust, add score 15.783907797979568 to doc_id 7384
Term: rust, add score 15.36327029130488 to doc_id 7760
Term: rust, add score 15.497215335170683 to doc_id 8381
Term: rust, add score 14.112967533767291 to doc_id 9292
Term: rust, add score 20.431324278615886 to doc_id 9889
Processing term: programming
Term: programming, add score 7.456932306459581 to doc_id 22

### None of the results is about Rust
Let's implement AND query.

In [147]:
def taat_and(query):
    accumulators = {}
    isNecessary:bool = False
    for term in get_tokens(query):
        isNecessary = term[0] == '+'
        if(isNecessary):
            print(f"Processing term: {term[1:]}")
            if term[1:] not in vocabulary_map:
                print("Term not present")
                return
            term_id = vocabulary_map[term[1:]]
        else:   
            print(f"Processing term: {term}")
            if term not in vocabulary_map:
                print("Term not present")
                continue
            else: 
                term_id = vocabulary_map[term]

        for doc_id, term_freq in zip(posting_lists[term_id], tf[term_id]):
            score = compute_bm25(term_id, doc_id, term_freq, idf, avgdl)
            if doc_id not in accumulators:
                accumulators[doc_id] = (0, 0.0) # number of matched terms, score
            (matches, cur_score) = accumulators[doc_id]
            accumulators[doc_id] = (matches+1, cur_score + score) 
    return sorted(accumulators.items(), key = lambda x: x[1], reverse = True) # sort by number of matches first

In [149]:
top_5 = taat_and("+Rust programming language")
print(len(top_5))
if(top_5 != None):
    print_top(top_5)

Processing term: rust
Processing term: programming
Processing term: language
333



Score: (2, 31.750792542965197)
BOOK : C Programming Language (2nd Edition) by Brian W. Kernighan, Dennis M. Ritchie. This is slightly more complex than a tutorial or a book to learn C. It is more of a reference. But is was written by the guys who invented C, so what can I say. It is the definite source for accurate information on the C language.... of another programming language, if not it will be hard for you to search Wikipedia for every word you don’t understand, but it would be useful. Check out the “Advanced Visual C# Programming” tutorial once you finish this one.
21/11/2011 · Want to learn a different language? Over the course of 24 episodes, our friend Bob Tabor from www.LearnVisualStudio.net will teach you the fundamentals of C# …... BOOK : C Programming Language (2nd Edition) by Brian W. Kernighan, Dennis M. Ritchie. This is slightly more complex than a tutorial or a book to learn C. It is mo

<br>

### Exercise

Modern search engines allows the use of operator "+" to force the presence of a term.
For example the query "Programming language +Rust" forces the presence of term "Rust". 

Are you able to modify the previous to allows operator "+"?

<br><br>

### Document-at-a-Time (DAAT)

<br><br>

#### Support operations with skipping


<img src="imgs/skipping.png" alt="alt text" width="700" />


In [151]:
class Skipping:
    def __init__(self, plist, skipping = 100):
        self.skipping = skipping
        self.plist = plist
        self.skips = plist[skipping-1::skipping] # the first element is the skipping-th one

class Operations:
    def __init__(self, skips):
        self.pos = 0
        self.skips = skips
        self.cost = 0  # Number of touched postings

    def get(self):
        return (self.pos, self.docId())

    def next(self):
        self.pos += 1
        return (self.pos, self.docId())

    def nextGEQ(self, target):
        skipping = self.skips.skipping
        #print("pos1:" + str(self.pos))
        while self.pos < len(self.skips.plist) and self.skips.plist[self.pos] < target:
            if self.pos % skipping == 0: # I can use skipping
                break
            self.pos += 1
            self.cost += 1 
        #print("pos2:" + str(self.pos))
        pos_skips = self.pos // skipping
        while pos_skips < len(self.skips.skips) and self.skips.skips[pos_skips] < target:
            pos_skips += 1
            self.cost += 1 
        #print("pos_skips:" + str(pos_skips))
        #print("cost:" + str(self.cost))
        #print("pos3:" + str(self.pos))
        self.pos = pos_skips * skipping
        #print("pos4:" + str(self.pos))
        while self.pos < len(self.skips.plist) and self.skips.plist[self.pos] < target:
            self.pos += 1
            self.cost += 1 
        #print("pos5:" + str(self.pos))

        return (self.pos, self.docId())

    def docId(self):
        if self.pos >= len(self.skips.plist):
            return None
        return self.skips.plist[self.pos]

    def get_cost(self):
        return self.cost

##### Example

In [112]:
l = [0, 10, 11, 20, 21, 22, 23, 28, 30, 35]
#    0   1   2   3   4   5   6   7   8   9

skips = Skipping(l, 5)

ops = Operations(skips)

print(skips.skips)

[21, 35]


In [115]:
ops.nextGEQ(23)

pos1:9
pos2:9
pos_skips:1
cost:9
pos3:9
pos4:5
pos5:6


(6, 23)

In [ ]:
ops.nextGEQ(30)

In [114]:
ops.nextGEQ(31)

pos1:6
pos2:9
pos_skips:1
cost:5
pos3:9
pos4:5
pos5:9


(9, 35)

In [ ]:
ops.nextGEQ(37)

<br><br>

#### AND query with skipping

In [116]:
def get_terms_posting_list(query, skipping):
    term_lists = []     # posting lists of the query terms
    curr_doc_ids = []   # the doc_ids currently pointed on the posting lists
    
    for term in get_tokens(query):
        if term not in vocabulary_map: 
            break
        term_id = vocabulary_map[term]

        curr_list = Operations( Skipping( posting_lists[term_id], skipping) ) #### Here Skipping is built ONLINE. In real life, this is done OFFLINE!
        term_lists.append( curr_list )
        _, doc_id = curr_list.get()
        curr_doc_ids.append( doc_id )  # doc_id at position 0
        
    return term_lists, curr_doc_ids

<img src="imgs/daat_and.png" alt="alt text" width="700" />


In [117]:
def AND_skipping(query, skipping = 100):
    
    term_lists, curr_doc_ids = get_terms_posting_list(query, skipping)
    
    r = []
    while True:
        max_doc_id = max(curr_doc_ids)
        arg_min, min_doc_id = min( enumerate(curr_doc_ids), key = lambda x: x[1]); # compare by doc_ids 

        doc_id = 0
        
        if max_doc_id == min_doc_id: # we have a result
            
            r.append( max_doc_id )
            _, doc_id = term_lists[0].next()
            curr_doc_ids[0] = doc_id
        else:
            _, doc_id = term_lists[arg_min].nextGEQ( max_doc_id ) 
            curr_doc_ids[arg_min] = doc_id

        if doc_id == None:
            break
            
    cost = sum(term_list.get_cost() for term_list in term_lists)
    return r, cost

In [152]:
query = "information retrieval python"

for skipping in [1, 5, 10, 15, 20, 50, 75, 100, 150]:
    r, cost = AND_skipping(query, skipping)
    print("skipping: {0:>3}\tcost: {1:>10}\tnumber of results: {2:>5}".format(skipping, cost, len(r)) )
print()

cost = 0
for term in get_tokens(query):
    if term in vocabulary_map: 
        term_id = vocabulary_map[term]
        cost += len(posting_lists[term_id])        
        print("term: {0:13}  n_postings: {1:>10}".format(term, len(posting_lists[term_id])))

print(f"\nWithout skipping the cost would be: {cost}")

skipping:   1	cost:       1261	number of results:     0
skipping:   5	cost:        316	number of results:     0
skipping:  10	cost:        257	number of results:     0
skipping:  15	cost:        256	number of results:     0
skipping:  20	cost:        290	number of results:     0
skipping:  50	cost:        482	number of results:     0
skipping:  75	cost:        624	number of results:     0
skipping: 100	cost:        801	number of results:     0
skipping: 150	cost:       1008	number of results:     0

term: information    n_postings:       1345
term: retrieval      n_postings:         15
term: python         n_postings:         13

Without skipping the cost would be: 1373


<br><br>

#### AND query with shortest lists first

<img src="imgs/taat_and.png" alt="alt text" width="700" />

In [ ]:
def AND_shortest_first(query, skipping = 100):
    
    term_list, curr_doc_ids = get_terms_posting_list(query)

    term_lists.sort(key = lambda x: len(x)) # sort by length

    cost = 0
    r = term_lists[0]
    for cur_list in term_lists[1:]:
        cur_list = Operations( Skipping( cur_list, skipping) ) #### Here Skipping is built ONLINE. In real life, this is done OFFLINE!
        cur_r = []
        for e in r:
            _, doc_id = cur_list.nextGEQ(e)
            if doc_id == None:
                break
            if e == doc_id:
                cur_r.append(e)
        cost += cur_list.get_cost()
        r = cur_r
    return r, cost

In [ ]:
query = "information retrieval python"

for skipping in [1, 5, 10, 15, 20, 50, 75, 100, 150]:
    r, cost = AND_shortest_first(query, skipping)
    print("skipping: {0:>3}\tcost: {1:>10}\tnumber of results: {2:>5}".format(skipping,cost, len(r)) )
print()

cost = 0
for term in get_tokens(query):
    if term in vocabulary_map: 
        term_id = vocabulary_map[term]
        cost += len(posting_lists[term_id])
        print("term: {0:13}  n_postings: {1:>10}".format(term, len(posting_lists[term_id])))

print(f"\nWithout skipping the cost would be: {cost}")

TypeError: 'Operations' object is not iterable

<br><br>

Lower cost! But more esperiments are needed in order to evaluate the fastest in practice!